#**LSTM(Long Short-Term Memory) Model**

In [ ]:
!pip install tensorflow pandas numpy scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### **Importing Libraries**

The necessary libraries were imported to handle data preprocessing, LSTM model building, training optimization, performance evaluation, and model persistence.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import joblib

###**Loading the Cleaned Agricultural Price Dataset**

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Cleaned_agriculture_price_dataset.csv')
df.head()


,price_date,state,district_name,market_name,commodity,variety,min_price,max_price,modal_price
0,2023-06-06,Maharashtra,Nashik,Lasalgaon(Niphad),Wheat,Maharashtra 2189,2172.0,2399.0,2300.0
1,2023-06-06,Uttar Pradesh,Bijnor,Chaandpur,Tomato,Hybrid,600.0,700.0,650.0
2,2023-06-06,Jammu & Kashmir,Jammu,Batote,Tomato,Other,1800.0,2200.0,2000.0
3,2023-06-06,Gujarat,Dahod,Dahod,Wheat,147 Average,2500.0,2700.0,2600.0
4,2023-06-06,Madhya Pradesh,Guna,Guna(F&V),Tomato,Other,350.0,530.0,410.0


**Inference:** The dataset provides detailed crop pricing information including dates, locations, commodity types, varieties, and their min, max, and modal prices.

###**Date Conversion, Sorting, and Handling Missing Values**

In [ ]:
df['price_date'] = pd.to_datetime(df['price_date'])

df = df.sort_values(by=['market_name', 'commodity', 'price_date'])

df = df.dropna(subset=['modal_price'])

**Inference:** The dataset was converted to datetime, sorted by market, commodity, and date, and rows with missing modal prices were removed for clean analysis.

### **Checking Dataset Columns**

In [ ]:
print(df.columns.tolist())

['price_date', 'state', 'district_name', 'market_name', 'commodity', 'variety', 'min_price', 'max_price', 'modal_price']


**Inference:** The dataset contains key columns including date, location details, commodity information, and price metrics necessary for analysis and forecasting.

###**Encoding Categorical Variables**

In [ ]:
label_encoders = {}
cat_cols = ['state', 'district_name', 'market_name', 'commodity']

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

**Inference:** Categorical columns were encoded into numerical values using LabelEncoder to prepare the data for LSTM modeling.


### **Scaling Target Variable and Preparing Features for LSTM**


In [ ]:
# Keep only modal_price for LSTM scaling
price_scaler = MinMaxScaler()
df['modal_price_scaled'] = price_scaler.fit_transform(df[['modal_price']])

# Categorical features (encoded integers)
cat_features = df[cat_cols].values

# Create final data array for LSTM (modal_price + categorical features)
features = np.hstack([df['modal_price_scaled'].values.reshape(-1,1), cat_features])


**Inference:** The modal prices were scaled using MinMaxScaler and combined with encoded categorical features to create the final input for the LSTM model.

### **Creating Time-Series Sequences for LSTM**

In [ ]:
def create_sequences(data, n_steps=15):
    X, y = [], []
    for i in range(n_steps, len(data)):
        X.append(data[i-n_steps:i])       # shape: (n_steps, features)
        y.append(data[i, 0])              # modal_price_scaled is first column
    return np.array(X), np.array(y)

n_steps = 15
X, y = create_sequences(features, n_steps)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (695198, 15, 5)
y shape: (695198,)


**Inference:** The input features were shaped into sequences for LSTM with 15 timesteps and 5 features, and the target variable was prepared accordingly.

### **Splitting Data into Training and Testing Sets**

In [ ]:
split = int(0.8 * len(X))

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

**Inference:** The dataset was split into 80% training and 20% testing sets to evaluate the LSTM model's performance.

###**Building and Compiling the LSTM Model**

In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),

    LSTM(32),
    Dropout(0.2),

    Dense(1)  # predicting modal_price_scaled
])

model.compile(
    optimizer='adam',
    loss='mse'
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 15, 64)         │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 15, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,369 (118.63 KB)

 Trainable params: 30,369 (118.63 KB)

 Non-trainable params: 0 (0.00 B)

**Inference:** The LSTM model was successfully built with two LSTM layers, dropout for regularization, and a dense output layer, totaling 30,369 trainable parameters.

### **Training the LSTM Model with Early Stopping**

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=15,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/15
7821/7821 ━━━━━━━━━━━━━━━━━━━━ 163s 20ms/step - loss: 0.0328 - val_loss: 0.0047
Epoch 2/15
7821/7821 ━━━━━━━━━━━━━━━━━━━━ 153s 20ms/step - loss: 0.0060 - val_loss: 0.0049
Epoch 3/15
7821/7821 ━━━━━━━━━━━━━━━━━━━━ 153s 20ms/step - loss: 0.0048 - val_loss: 0.0113
Epoch 4/15
7821/7821 ━━━━━━━━━━━━━━━━━━━━ 153s 19ms/step - loss: 0.0045 - val_loss: 0.0076
Epoch 5/15
7821/7821 ━━━━━━━━━━━━━━━━━━━━ 202s 20ms/step - loss: 0.0043 - val_loss: 0.0092
Epoch 6/15
7821/7821 ━━━━━━━━━━━━━━━━━━━━ 162s 21ms/step - loss: 0.0042 - val_loss: 0.0121


**Inference:** The LSTM model training shows decreasing training loss, while validation loss fluctuates slightly, indicating effective learning with minor overfitting.

### **Evaluating the LSTM Model Performance**

In [ ]:
y_pred_scaled = model.predict(X_test)

# Inverse scale modal_price
y_test_inv = price_scaler.inverse_transform(y_test.reshape(-1,1))[:,0]
y_pred_inv = price_scaler.inverse_transform(y_pred_scaled)[:,0]

mae = mean_absolute_error(y_test_inv, y_pred_inv)
rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
r2 = r2_score(y_test_inv, y_pred_inv)

print(f"LSTM MAE  : {mae:.2f}")
print(f"LSTM RMSE : {rmse:.2f}")
print(f"LSTM R²   : {r2:.3f}")


4345/4345 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step
LSTM MAE  : 261.89
LSTM RMSE : 414.95
LSTM R²   : 0.884


**Inference:** The LSTM model achieved MAE, RMSE, and R² values indicating it can predict crop prices with reasonable accuracy on the test set.

### **Saving the Trained LSTM Model and Label Encoders**

In [ ]:
rf_model = joblib.dump(model,'/content/drive/MyDrive/lstm_model.pkl')
joblib.dump(label_encoders, '/content/drive/MyDrive/lstm_label_encoders.pkl')

['/content/drive/MyDrive/lstm_label_encoders.pkl']

**Inference:** The label encoders used for categorical features were saved as a .pkl file to ensure consistent encoding during future predictions.

### **Function to Encode New Input Data Using Saved Label Encoders**

In [ ]:
def encode_inputs(state, district, market, commodity):
    try:
        return {
            'state': label_encoders['state'].transform([state])[0],
            'district_name': label_encoders['district_name'].transform([district])[0],
            'market_name': label_encoders['market_name'].transform([market])[0],
            'commodity': label_encoders['commodity'].transform([commodity])[0]
        }
    except ValueError:
        raise ValueError("❌ Input not found in training data. Please enter valid values.")


**Inference:** This function encodes new input values into numerical form using the saved label encoders, ensuring consistency with the training data.


###**Function to Predict Future Crop Prices Using the Trained LSTM Model**

In [ ]:
def predict_future_prices_lstm(
    df, model, state, district, market, commodity, days, n_steps, price_scaler
):
    # Encode input
    encoded = encode_inputs(state, district, market, commodity)

    # Filter data for selected series
    series = df[
        (df['state'] == encoded['state']) &
        (df['district_name'] == encoded['district_name']) &
        (df['market_name'] == encoded['market_name']) &
        (df['commodity'] == encoded['commodity'])
    ].sort_values('price_date')

    if len(series) < n_steps:
        return "❌ Not enough historical data for LSTM prediction."

    # Prepare last sequence
    cat_values = series[['state','district_name','market_name','commodity']].values
    price_values = series['modal_price_scaled'].values.reshape(-1,1)

    last_seq = np.hstack([price_values[-n_steps:], cat_values[-n_steps:]])
    last_seq = last_seq.reshape(1, n_steps, last_seq.shape[1])

    # Set last_date to today (only date, no timestamp)
    last_date = pd.Timestamp.today().date()

    results = []

    for _ in range(days):
        # Predict next scaled price
        pred_scaled = model.predict(last_seq, verbose=0)[0][0]
        # Inverse transform to original price
        pred_price = price_scaler.inverse_transform([[pred_scaled]])[0][0]

        # Next date
        next_date = last_date + pd.Timedelta(days=1)
        results.append({'date': next_date, 'predicted_price': round(float(pred_price),2)})

        # Update sequence (shift left + append new prediction + same categorical features)
        new_row = last_seq[0, -1, :].copy()
        new_row[0] = pred_scaled
        last_seq = np.roll(last_seq, -1, axis=1)
        last_seq[0, -1, :] = new_row

        last_date = next_date

    return pd.DataFrame(results)


**Inference:** This function generates future crop price predictions for a specified number of days by iteratively using the LSTM model and maintaining the input sequence.


### **Predicting and Displaying Future Crop Prices Based on User Input**

In [ ]:
market = input("Enter market name: ")
district = input("Enter district name: ")
state = input("Enter state: ")
commodity = input("Enter commodity: ")
days = int(input("Enter number of days to predict: "))

future_prices = predict_future_prices_lstm(
    df=df,
    model=model,
    state=state,
    district=district,
    market=market,
    commodity=commodity,
    days=days,
    n_steps=n_steps,
    price_scaler=price_scaler
)

future_prices


Enter market name: Batote
Enter district name: Jammu
Enter state: Jammu & Kashmir
Enter commodity: Tomato
Enter number of days to predict: 7


,date,predicted_price
0,2025-12-24,2018.41
1,2025-12-25,2026.09
2,2025-12-26,2018.92
3,2025-12-27,2008.71
4,2025-12-28,1998.34
5,2025-12-29,1988.76
6,2025-12-30,1980.15


**Inference:** The LSTM model successfully predicted the next 7 days of crop prices for Tomato in Batote, showing gradual fluctuations in forecasted values.

 **Conclusion:**

🔹 Data Cleaning & Processing
The dataset was cleaned, sorted by time, encoded, and scaled to ensure it was suitable for time-series forecasting using LSTM.

🔹 Exploratory Data Analysis
EDA helped in understanding crop price trends, variations across markets, and temporal patterns influencing price fluctuations.

🔹 Model Building
An LSTM-based deep learning model was developed to capture long-term dependencies in historical crop price data.

🔹 Model Evaluation
The model performance was evaluated using MAE, RMSE, and R² metrics, showing reasonable prediction accuracy on unseen data.

🔹 Key Takeaway
LSTM effectively learned sequential price patterns and produced stable short-term crop price forecasts.

🔹 Overall
The LSTM model plays a crucial role in this project by enabling reliable crop price forecasting, which can support farmers, traders, and policymakers in better market decision-making.